In [1]:
from pyspark.sql import SparkSession


# 0. Initialisation de la session Spark
spark = SparkSession.builder \
    .appName("Telco_Customer_Churn_EDA") \
    .getOrCreate()

# 1. Chargement du fichier de données (Format Parquet)
df = spark.read.parquet('data/yellow_tripdata_2025-02.parquet')

# 2. Analyse de la structure (Taille et types de données)
print(f"Taille du dataset : ({df.count()}, {len(df.columns)})")
print("-" * 40)
df.printSchema()

# 3. Affichage des 5 premières lignes pour un aperçu
df.show(5, truncate=False)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/30 12:22:37 WARN Utils: Your hostname, ThomasVIVOBOOK, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/30 12:22:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/30 12:22:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/30 12:22:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Taille du dataset : (3577543, 20)
----------------------------------------
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [2]:
# On exclut volontairement les "ID" qui sont des catégories
cols_numeriques = [
    'passenger_count', 'trip_distance', 'fare_amount', 
    'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 
    'improvement_surcharge', 'total_amount', 
    'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee'
]

print("--- Statistiques des colonnes numériques ---")
# Le .summary() calcule les quartiles (25%, 50%, 75%) utiles pour voir l'asymétrie
df.select(cols_numeriques).summary("count", "mean", "min", "25%", "50%", "75%", "max").show()

--- Statistiques des colonnes numériques ---


26/04/30 12:22:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+-----------------+-----------------+------------------+-------------------+-----------------+------------------+---------------------+------------------+--------------------+-------------------+------------------+
|summary|   passenger_count|    trip_distance|      fare_amount|             extra|            mta_tax|       tip_amount|      tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|        Airport_fee|cbd_congestion_fee|
+-------+------------------+-----------------+-----------------+------------------+-------------------+-----------------+------------------+---------------------+------------------+--------------------+-------------------+------------------+
|  count|           2770606|          3577543|          3577543|           3577543|            3577543|          3577543|           3577543|              3577543|           3577543|             2770606|            2770606|           3577543|
|   mean|1.2760276993553035|6.02